# Regression & Logistic Models — Interview Questions

This notebook covers the most common interview questions for:
- **Linear Regression** (Simple, Multiple, Ridge, Lasso, ElasticNet)
- **Logistic Regression** (Binary, Multiclass)

Each question includes a concise answer and a small coding example where applicable.

---

## Contents

| # | Section |
|---|---------|
| 1 | Linear Regression Interview Questions |
| 2 | Regularization Interview Questions (Ridge / Lasso / ElasticNet) |
| 3 | Logistic Regression Interview Questions |
| 4 | Evaluation Metrics Interview Questions |
| 5 | Quick Cheat Sheet |

In [ ]:
# ── Setup — run this first ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification, fetch_california_housing, load_breast_cancer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Setup complete.')

---
# Part 1 — Linear Regression Interview Questions

## Q1. What is Linear Regression? What problem does it solve?

**Answer:**
Linear Regression finds the best-fit straight line through data to predict a **continuous output** from one or more input features.

It solves **regression problems** — whenever you need a number as output (house price, temperature, salary).

The model learns two things:
- **θ₀ (intercept):** value of Y when all X = 0
- **θ₁…θₙ (slopes):** how much Y changes per unit increase in each feature

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n$$

**Training** = find θ values that minimize the Mean Squared Error (MSE) between predictions and actuals.

In [ ]:
# Q1 — Minimal linear regression example
X = np.array([[1000], [1500], [2000], [2500], [3000]])  # house size (sqft)
y = np.array([200, 300, 400, 500, 600])                  # price (k)

model = LinearRegression()
model.fit(X, y)

print(f'Intercept (θ₀): {model.intercept_:.2f}')
print(f'Slope (θ₁):     {model.coef_[0]:.4f}  → every extra sqft adds {model.coef_[0]:.4f}k to price')
print(f'Predict 1800 sqft: {model.predict([[1800]])[0]:.1f}k')

## Q2. What are the assumptions of Linear Regression?

**Answer — 7 Assumptions:**

| # | Assumption | What to check |
|---|------------|---------------|
| 1 | **Linearity** | Scatter plot of X vs Y should look linear |
| 2 | **Independence of errors** | Each data point is independent (no autocorrelation) |
| 3 | **Homoscedasticity** | Residuals have constant variance (no funnel shape in residual plot) |
| 4 | **Normality of errors** | Residuals are normally distributed (check Q-Q plot) |
| 5 | **No multicollinearity** | Features are not highly correlated with each other (check VIF) |
| 6 | **No autocorrelation** | Errors don't correlate over time (Durbin-Watson test) |
| 7 | **No outliers** | Extreme points can distort the line significantly |

**Key interview tip:** If assumptions are violated → residual plots will show patterns (not random clouds).

In [ ]:
# Q2 — Check assumptions visually: residual plot + Q-Q plot
from scipy import stats

housing = fetch_california_housing()
X_h = housing.data[:, 0].reshape(-1, 1)  # MedInc feature only
y_h = housing.target

m = LinearRegression().fit(X_h, y_h)
residuals = y_h - m.predict(X_h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual plot — should look like a random cloud around 0
axes[0].scatter(m.predict(X_h), residuals, alpha=0.2, color='steelblue')
axes[0].axhline(0, color='red', lw=1.5, linestyle='--')
axes[0].set(xlabel='Predicted', ylabel='Residuals', title='Residual Plot\n(should be random cloud — no pattern)')

# Q-Q plot — residuals should follow the diagonal for normality
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot\n(points on diagonal = normally distributed residuals)')

plt.tight_layout()
plt.show()
print('Residual plot: look for no pattern (random cloud = good)')
print('Q-Q plot: points hugging diagonal = normality assumption holds')

## Q3. What is the cost function in Linear Regression? Why MSE and not MAE?

**Answer:**

The cost function is **MSE (Mean Squared Error)**:
$$J(\theta) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

**Why MSE over MAE?**

| Reason | Explanation |
|--------|-------------|
| **Differentiable** | MSE has a smooth gradient everywhere → gradient descent works cleanly |
| **MAE is not smooth at 0** | The absolute value function has a kink at 0 — gradient is undefined |
| **Penalizes large errors more** | MSE squares errors, making big mistakes very costly → model avoids them |
| **Unique minimum** | MSE is convex (bowl shape) → gradient descent always finds the global minimum |

**When MAE is preferred:** when data has outliers you don't want to overweight.

In [ ]:
# Q3 — Show MSE vs MAE behavior on outliers
y_true    = np.array([100, 200, 300, 400, 500])
y_pred    = np.array([110, 195, 310, 390, 510])   # small errors
y_outlier = np.array([110, 195, 310, 390, 900])   # one big outlier

for label, pred in [('Normal predictions', y_pred), ('With outlier', y_outlier)]:
    mae  = mean_absolute_error(y_true, pred)
    mse  = mean_squared_error(y_true, pred)
    rmse = np.sqrt(mse)
    print(f'{label}:')
    print(f'  MAE  = {mae:.1f}   (outlier impact: linear)')
    print(f'  MSE  = {mse:.1f}   (outlier impact: squared!)')
    print(f'  RMSE = {rmse:.1f}')
    print()

## Q4. What is Gradient Descent? How does it optimize Linear Regression?

**Answer:**

Gradient Descent is an **iterative optimization algorithm** that finds the minimum of the cost function by repeatedly updating parameters in the direction of steepest descent.

**Update rule:**
$$\theta_j := \theta_j - \alpha \frac{\partial J}{\partial \theta_j}$$

| Term | Meaning |
|------|--------|
| α (alpha) | Learning rate — step size |
| ∂J/∂θⱼ | Gradient — slope of cost function |
| Gradient = + | θ too large → decrease it |
| Gradient = − | θ too small → increase it |
| Gradient = 0 | At minimum → stop |

**Types of Gradient Descent:**
- **Batch GD:** uses all samples per step — slow but accurate
- **Stochastic GD (SGD):** uses 1 sample per step — fast but noisy
- **Mini-batch GD:** uses small batches — best of both worlds

In [ ]:
# Q4 — Gradient Descent from scratch for simple linear regression
X_gd = np.array([1000, 1500, 2000, 2500, 3000], dtype=float)
y_gd = np.array([200,  300,  400,  500,  600],  dtype=float)

theta0, theta1 = 0.0, 0.0  # starting guess
alpha = 1e-7                # learning rate
n = len(X_gd)

print('Iteration | theta1   | MSE')
print('-' * 35)
for iteration in range(1, 6001):
    y_hat    = theta0 + theta1 * X_gd
    error    = y_hat - y_gd
    grad_t0  = (2/n) * error.sum()
    grad_t1  = (2/n) * (error * X_gd).sum()
    theta0  -= alpha * grad_t0
    theta1  -= alpha * grad_t1
    if iteration in [1, 100, 500, 1000, 3000, 6000]:
        mse = np.mean(error**2)
        print(f'{iteration:9d} | {theta1:.6f} | {mse:.4f}')

print(f'\nFinal: θ₀={theta0:.2f}, θ₁={theta1:.6f}')
print(f'sklearn: θ₀={LinearRegression().fit(X_gd.reshape(-1,1), y_gd).intercept_:.2f},',
      f'θ₁={LinearRegression().fit(X_gd.reshape(-1,1), y_gd).coef_[0]:.6f}')

## Q5. What is Multicollinearity? How do you detect and fix it?

**Answer:**

**Multicollinearity** = when two or more features are highly correlated with each other.

**Problem:** The model can't distinguish which feature is responsible for the output → coefficients become unstable and unreliable.

**Detection:**
- **Correlation matrix** — look for pairs with |correlation| > 0.8
- **VIF (Variance Inflation Factor)** — VIF > 5–10 signals multicollinearity

**Fix:**
- Remove one of the correlated features
- Use **Ridge Regression** (handles multicollinearity automatically)
- Apply PCA to combine correlated features

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
where R²ⱼ = R² when predicting feature j from all other features.

In [ ]:
# Q5 — Detect multicollinearity with correlation matrix and VIF
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)

# Correlation matrix
print('Correlation matrix (|value| > 0.5 = potential multicollinearity):')
corr = df.corr().round(2)
print(corr)

# VIF — manual calculation
print('\nVIF per feature (> 5 = multicollinearity concern):')
from sklearn.linear_model import LinearRegression
vif_data = []
for i, col in enumerate(df.columns):
    X_vif = df.drop(columns=[col]).values
    y_vif = df[col].values
    r2 = LinearRegression().fit(X_vif, y_vif).score(X_vif, y_vif)
    vif = 1 / (1 - r2) if r2 < 1 else float('inf')
    vif_data.append({'Feature': col, 'VIF': round(vif, 2)})
print(pd.DataFrame(vif_data).to_string(index=False))

## Q6. What is Overfitting vs Underfitting in regression?

**Answer:**

| | Underfitting | Good Fit | Overfitting |
|--|-------------|----------|-------------|
| **What** | Model too simple — misses patterns | Captures true signal | Model too complex — memorizes noise |
| **Train error** | High | Low | Very low |
| **Test error** | High | Low | High |
| **Gap (test-train)** | Small | Small | Large |
| **Fix** | More features, complex model | — | Regularization, fewer features, more data |

**How to detect:** train RMSE vs test RMSE — large gap = overfitting.

In [ ]:
# Q6 — Demonstrate overfitting vs underfitting using polynomial regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Generate noisy sine wave data
X_poly = np.linspace(0, 2*np.pi, 30).reshape(-1, 1)
y_poly = np.sin(X_poly).ravel() + np.random.normal(0, 0.2, 30)
X_test_p = np.linspace(0, 2*np.pi, 100).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 4, 15]
titles  = ['Underfitting (degree=1)', 'Good Fit (degree=4)', 'Overfitting (degree=15)']

for ax, deg, title in zip(axes, degrees, titles):
    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    pipe.fit(X_poly, y_poly)
    y_pred_train = pipe.predict(X_poly)
    y_pred_test  = pipe.predict(X_test_p)
    train_rmse = np.sqrt(mean_squared_error(y_poly, y_pred_train))
    ax.scatter(X_poly, y_poly, color='steelblue', label='Data', zorder=5)
    ax.plot(X_test_p, y_pred_test, color='red', lw=2, label='Model')
    ax.set_title(f'{title}\nTrain RMSE={train_rmse:.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
# Part 2 — Regularization Interview Questions (Ridge / Lasso / ElasticNet)

## Q7. What is Regularization? Why do we need it?

**Answer:**

Regularization adds a **penalty term** to the cost function to discourage the model from learning overly large coefficients.

$$J(\theta) = \underbrace{\text{MSE}}_\text{fit the data} + \underbrace{\lambda \cdot \text{penalty}}_\text{keep weights small}$$

**Why?** Without regularization, on noisy or high-dimensional data:
- Model learns very large weights to perfectly fit training data
- These large weights make predictions extremely sensitive to tiny input changes
- Result: great on training set, terrible on new data (overfitting)

**Types:**
| Name | Penalty | Effect |
|------|---------|--------|
| Ridge (L2) | λ Σwⱼ² | Shrinks all weights toward 0, none exactly 0 |
| Lasso (L1) | λ Σ\|wⱼ\| | Shrinks weights, some become exactly 0 (feature selection) |
| ElasticNet | L1 + L2 | Both effects combined |

## Q8. What is the difference between Ridge and Lasso? When would you use each?

**Answer:**

| | Ridge (L2) | Lasso (L1) |
|--|------------|------------|
| **Penalty** | λ Σwⱼ² | λ Σ\|wⱼ\| |
| **Zeros out weights?** | No — shrinks toward 0 | Yes — some become exactly 0 |
| **Feature selection** | No | Yes (automatic) |
| **Correlated features** | Handles well (shares weight) | Picks one, drops rest |
| **When to use** | All features matter; multicollinearity present | Many irrelevant features; want sparse model |

**Why does Lasso reach 0 but Ridge doesn't?**
- Ridge penalty w² has a smooth curve — never creates an incentive to go to exactly 0
- Lasso penalty |w| has a sharp corner (kink) at 0 — optimizer snaps small weights there

**Interview trick question:** "Can Ridge give zero coefficients?" → Answer: No, not exactly zero, only very close.

In [ ]:
# Q8 — Side-by-side: Ridge vs Lasso coefficients
housing = fetch_california_housing()
X_h = housing.data
y_h = housing.target
X_tr, X_te, y_tr, y_te = train_test_split(X_h, y_h, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

models = {
    'Linear' : LinearRegression(),
    'Ridge'  : Ridge(alpha=1.0),
    'Lasso'  : Lasso(alpha=0.1)
}

print(f'{"Feature":<12}', end='')
for name in models:
    print(f'{name:>12}', end='')
print()
print('-' * 48)

coefs = {}
for name, m in models.items():
    m.fit(X_tr_s, y_tr)
    coefs[name] = m.coef_

for i, feat in enumerate(housing.feature_names):
    print(f'{feat:<12}', end='')
    for name in models:
        val = coefs[name][i]
        tag = ' ← 0!' if abs(val) < 1e-6 else ''
        print(f'{val:>11.4f}{tag}', end='')
    print()

print()
print('Lasso zeros out features (← 0!), Ridge never does.')

## Q9. What is ElasticNet? When is it better than Ridge or Lasso?

**Answer:**

ElasticNet combines **L1 + L2** penalties:

$$J(\theta) = \text{MSE} + \lambda \left[ \alpha \sum|w_j| + (1-\alpha)\sum w_j^2 \right]$$

- `l1_ratio = 1` → pure Lasso
- `l1_ratio = 0` → pure Ridge
- `l1_ratio = 0.5` → 50/50 mix

**When ElasticNet wins over Lasso:**
When you have **correlated features** — Lasso arbitrarily picks one and drops the rest. ElasticNet's L2 part distributes weight across the correlated group fairly.

**When ElasticNet wins over Ridge:**
When you have **many irrelevant features** — Ridge can't eliminate them; ElasticNet's L1 part zeros them out.

**Rule of thumb:** Default to ElasticNet when you have > 100 features and don't know which regularizer to pick.

In [ ]:
# Q9 — Effect of l1_ratio in ElasticNet
ratios = [0.0, 0.25, 0.5, 0.75, 1.0]
print(f'{"l1_ratio":<12} {"zeros":<10} {"R²":<10} {"RMSE"}')
print('-' * 45)
for ratio in ratios:
    m = ElasticNet(alpha=0.1, l1_ratio=ratio, max_iter=10000)
    m.fit(X_tr_s, y_tr)
    pred   = m.predict(X_te_s)
    zeros  = (m.coef_ == 0).sum()
    r2     = r2_score(y_te, pred)
    rmse   = np.sqrt(mean_squared_error(y_te, pred))
    label  = '← pure Ridge' if ratio == 0 else ('← pure Lasso' if ratio == 1 else '')
    print(f'{ratio:<12} {zeros:<10} {r2:<10.4f} {rmse:.4f}  {label}')

## Q10. What is the role of the regularization parameter λ (alpha)? How do you choose it?

**Answer:**

λ controls the **strength of the penalty**:

| λ value | Effect |
|---------|--------|
| λ = 0 | No regularization — same as plain linear regression |
| λ small | Slight shrinkage — mostly fits data |
| λ large | Heavy shrinkage — coefficients → 0 |
| λ → ∞ | All coefficients = 0 — predicts the mean |

**How to choose λ:**
Use **Cross-Validation** — try a range of λ values (e.g., 0.001 to 1000 on log scale), pick the one that gives lowest validation error.

sklearn has `RidgeCV` and `LassoCV` that do this automatically.

In [ ]:
# Q10 — Use RidgeCV to automatically select best lambda
from sklearn.linear_model import RidgeCV, LassoCV

alphas = np.logspace(-3, 3, 50)  # try 50 values from 0.001 to 1000

ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_tr_s, y_tr)

lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000)
lasso_cv.fit(X_tr_s, y_tr)

print(f'Best lambda for Ridge : {ridge_cv.alpha_:.4f}')
print(f'Best lambda for Lasso : {lasso_cv.alpha_:.4f}')

# Show how RMSE changes with lambda
test_alphas = np.logspace(-3, 2, 40)
rmses = [np.sqrt(mean_squared_error(y_te, Ridge(alpha=a).fit(X_tr_s, y_tr).predict(X_te_s)))
         for a in test_alphas]

plt.figure(figsize=(8, 4))
plt.semilogx(test_alphas, rmses, 'b-o', markersize=4)
plt.axvline(ridge_cv.alpha_, color='red', linestyle='--', label=f'Best λ={ridge_cv.alpha_:.3f}')
plt.xlabel('Lambda (log scale)')
plt.ylabel('Test RMSE')
plt.title('Ridge: How test RMSE changes with λ')
plt.legend()
plt.tight_layout()
plt.show()

---
# Part 3 — Logistic Regression Interview Questions

## Q11. What is Logistic Regression? Why is it called "regression" if it classifies?

**Answer:**

Logistic Regression predicts the **probability** that an input belongs to a class (0 or 1).

**Why "regression"?** Because internally it computes a linear combination of features (like linear regression), then squashes that number into a probability using the **sigmoid function**:

$$P(y=1 | x) = \sigma(\theta^T x) = \frac{1}{1 + e^{-z}} \quad \text{where} \quad z = \theta_0 + \theta_1 x_1 + \cdots$$

**Decision boundary:**
- P(y=1) ≥ 0.5 → predict class 1
- P(y=1) < 0.5 → predict class 0

**Sigmoid properties:**
- Output always between 0 and 1
- σ(0) = 0.5 exactly
- As z → +∞, σ(z) → 1
- As z → −∞, σ(z) → 0

In [ ]:
# Q11 — Visualize the sigmoid function
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid, 'b-', lw=2.5, label='σ(z) = 1 / (1 + e^-z)')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary (0.5)')
plt.axvline(0, color='gray', linestyle=':', alpha=0.6)
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.15, color='green', label='Predict class 1')
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.15, color='red',   label='Predict class 0')
plt.xlabel('z (linear combination of features)')
plt.ylabel('Probability')
plt.title('Sigmoid Function — Maps any number to (0, 1)')
plt.legend()
plt.ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print('σ(-8) =', round(1/(1+np.exp(8)), 5), '← very close to 0')
print('σ(0)  =', round(1/(1+np.exp(0)), 5), '← exactly 0.5')
print('σ(+8) =', round(1/(1+np.exp(-8)), 5), '← very close to 1')

## Q12. What cost function does Logistic Regression use? Why not MSE?

**Answer:**

Logistic Regression uses **Binary Cross-Entropy (Log Loss)**:

$$J(\theta) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1-y_i) \log(1-\hat{p}_i)\right]$$

**Why not MSE?**

| | MSE with sigmoid | Log Loss |
|--|-----------------|----------|
| **Shape** | Non-convex — multiple local minima | Convex — one global minimum |
| **Gradient descent** | May get stuck in local minimum | Always finds global minimum |
| **Gradient size** | Vanishing gradients for confident wrong predictions | Large gradients when confident and wrong |

**Intuition for log loss:**
- Actual y=1, predicted p=0.99 → loss = −log(0.99) ≈ 0.01 (small — correct prediction)
- Actual y=1, predicted p=0.01 → loss = −log(0.01) ≈ 4.6 (huge — confident wrong prediction)

In [ ]:
# Q12 — Show log loss intuition
probs = np.linspace(0.001, 0.999, 200)
loss_y1 = -np.log(probs)      # loss when actual = 1
loss_y0 = -np.log(1 - probs)  # loss when actual = 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(probs, loss_y1, 'b-', lw=2)
axes[0].set(xlabel='Predicted probability', ylabel='Loss',
            title='Actual = 1\n(low loss when prediction is high)')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.5)

axes[1].plot(probs, loss_y0, 'r-', lw=2)
axes[1].set(xlabel='Predicted probability', ylabel='Loss',
            title='Actual = 0\n(low loss when prediction is low)')
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Log Loss — Penalizes confident wrong predictions heavily', fontweight='bold')
plt.tight_layout()
plt.show()

# Example: actual=1
print('When actual=1:')
for p in [0.99, 0.8, 0.5, 0.2, 0.01]:
    loss = -np.log(p)
    print(f'  predicted={p:.2f} → loss={loss:.4f}')

## Q13. What is the difference between Linear Regression and Logistic Regression?

**Answer:**

| | Linear Regression | Logistic Regression |
|--|-------------------|---------------------|
| **Output** | Continuous number (e.g., 250k) | Probability (0 to 1) → class label |
| **Task** | Regression | Classification |
| **Output function** | Identity (ŷ = θᵀx) | Sigmoid σ(θᵀx) |
| **Cost function** | MSE | Binary Cross-Entropy |
| **Decision** | Not applicable | P ≥ 0.5 → class 1 |
| **Assumptions** | Linearity of X vs Y | Linearity of X vs log-odds |
| **Output range** | −∞ to +∞ | 0 to 1 |

**Key point:** Both compute `z = θ₀ + θ₁x₁ + … + θₙxₙ`. Logistic adds a sigmoid on top to squash z into a probability.

In [ ]:
# Q13 — Logistic Regression on breast cancer dataset
cancer = load_breast_cancer()
X_c = cancer.data
y_c = cancer.target

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_c_s = sc.fit_transform(X_tr_c)
X_te_c_s  = sc.transform(X_te_c)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_tr_c_s, y_tr_c)

y_pred_c = log_reg.predict(X_te_c_s)
y_prob_c = log_reg.predict_proba(X_te_c_s)[:, 1]

print('Logistic Regression — Breast Cancer Classification')
print(f'Accuracy : {accuracy_score(y_te_c, y_pred_c):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_te_c, y_prob_c):.4f}')
print()
print('Classification Report:')
print(classification_report(y_te_c, y_pred_c, target_names=cancer.target_names))

## Q14. What is the Confusion Matrix? Explain Precision, Recall, F1-Score.

**Answer:**

```
               Predicted
               Positive  Negative
Actual  Pos |   TP    |    FN   |
        Neg |   FP    |    TN   |
```

| Term | Formula | Meaning |
|------|---------|--------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Overall correct predictions |
| **Precision** | TP/(TP+FP) | Of all predicted positives, how many are truly positive? |
| **Recall** | TP/(TP+FN) | Of all actual positives, how many did we catch? |
| **F1-Score** | 2×P×R/(P+R) | Harmonic mean of precision and recall |

**When to use which:**
- **Precision matters** → Spam detection (don't want to mislabel real email as spam)
- **Recall matters** → Cancer detection (don't want to miss a real cancer case)
- **F1 matters** → Imbalanced classes where both FP and FN are costly

In [ ]:
# Q14 — Confusion matrix visualization
import matplotlib.patches as mpatches

cm = confusion_matrix(y_te_c, y_pred_c)
tn, fp, fn, tp = cm.ravel()

precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1        = 2 * precision * recall / (precision + recall)
accuracy  = (tp + tn) / cm.sum()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]}', ha='center', va='center',
                fontsize=18, fontweight='bold', color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: Malignant', 'Pred: Benign'])
ax.set_yticklabels(['Actual: Malignant', 'Actual: Benign'])
ax.set_title('Confusion Matrix — Breast Cancer', fontweight='bold')
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')
print(f'Precision = {tp}/({tp}+{fp}) = {precision:.4f}  → of predicted malignant, {precision*100:.1f}% truly are')
print(f'Recall    = {tp}/({tp}+{fn}) = {recall:.4f}    → caught {recall*100:.1f}% of all malignant cases')
print(f'F1-Score  = {f1:.4f}')
print(f'Accuracy  = {accuracy:.4f}')

## Q15. What is the ROC Curve and AUC? What does AUC=0.5 mean?

**Answer:**

**ROC (Receiver Operating Characteristic) Curve** plots:
- X-axis: False Positive Rate (FPR) = FP / (FP + TN)
- Y-axis: True Positive Rate / Recall = TP / (TP + FN)

At each probability threshold (0 → 1), you get a different FPR/TPR point — connecting them gives the curve.

**AUC (Area Under Curve):**

| AUC | Meaning |
|-----|---------|
| 1.0 | Perfect model |
| 0.9+ | Excellent |
| 0.7–0.9 | Good |
| 0.5 | **Random guess** — model has no discriminating power |
| < 0.5 | Worse than random (flip predictions!) |

**AUC is threshold-independent** — it measures how well the model ranks positives above negatives regardless of the cutoff you choose.

**When to use AUC over accuracy?** When classes are imbalanced — accuracy can be misleading (99% accuracy if 99% is class 0).

In [ ]:
# Q15 — ROC curve comparison: good model vs random
fpr, tpr, _ = roc_curve(y_te_c, y_prob_c)
auc = roc_auc_score(y_te_c, y_prob_c)

# Random predictor
random_probs = np.random.uniform(0, 1, len(y_te_c))
fpr_rand, tpr_rand, _ = roc_curve(y_te_c, random_probs)
auc_rand = roc_auc_score(y_te_c, random_probs)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr,           lw=2.5, color='blue',  label=f'Logistic Regression (AUC={auc:.3f})')
plt.plot(fpr_rand, tpr_rand, lw=1.5, color='orange',linestyle='--', label=f'Random (AUC={auc_rand:.3f})')
plt.plot([0,1], [0,1],       lw=1,   color='gray',  linestyle=':', label='Random = diagonal')
plt.fill_between(fpr, tpr, alpha=0.1, color='blue')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('ROC Curve — Logistic Regression vs Random')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Logistic AUC = {auc:.4f}  (closer to 1 = better)')
print(f'Random   AUC ≈ 0.5  (diagonal line = no discriminating power)')

## Q16. What is class imbalance? How do you handle it in Logistic Regression?

**Answer:**

**Class imbalance** = when one class vastly outnumbers the other (e.g., 99% non-fraud, 1% fraud).

**Problem:** Model learns to predict the majority class all the time → 99% accuracy but catches 0% fraud.

**Fixes:**

| Fix | How it works |
|-----|--------------|
| `class_weight='balanced'` | sklearn auto-weights classes inversely to frequency |
| **SMOTE** | Synthetically generates minority class samples |
| **Undersampling** | Reduce majority class samples |
| **Change threshold** | Lower decision boundary from 0.5 to 0.3 to catch more positives |
| **Use AUC not accuracy** | AUC is imbalance-robust |

**Always use:** `class_weight='balanced'` as a first easy fix.

In [ ]:
# Q16 — Imbalanced data: class_weight impact
# Create imbalanced dataset: 950 class 0, 50 class 1
X_imb, y_imb = make_classification(
    n_samples=1000, weights=[0.95, 0.05],
    n_features=10, random_state=42
)
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_imb, y_imb, test_size=0.2, stratify=y_imb, random_state=42)

for label, cw in [('No class_weight', None), ('class_weight=balanced', 'balanced')]:
    m = LogisticRegression(class_weight=cw, max_iter=1000)
    m.fit(X_tr_i, y_tr_i)
    pred = m.predict(X_te_i)
    prob = m.predict_proba(X_te_i)[:, 1]
    acc  = accuracy_score(y_te_i, pred)
    auc  = roc_auc_score(y_te_i, prob)
    rec  = (pred[y_te_i==1] == 1).mean()  # recall on minority class
    print(f'{label}:')
    print(f'  Accuracy={acc:.4f}  AUC={auc:.4f}  Minority Recall={rec:.4f}')
    print()

print('Lesson: High accuracy can hide poor recall on the minority class.')
print('AUC and Recall are better metrics for imbalanced data.')

## Q17. How does Logistic Regression handle multiclass classification?

**Answer:**

Logistic Regression handles multiclass via two strategies:

**1. One-vs-Rest (OvR):**
- Train one binary classifier per class: "Is this class A vs everything else?"
- For K classes → K classifiers
- Predict: class with highest probability wins
- `multi_class='ovr'`

**2. Softmax (Multinomial):**
- Train one model that outputs probabilities for all K classes simultaneously
- Uses softmax function instead of sigmoid: $P(y=k) = \frac{e^{z_k}}{\sum_j e^{z_j}}$
- Probabilities for all classes sum to 1
- `multi_class='multinomial'`

**Comparison:**
| | OvR | Multinomial/Softmax |
|--|-----|---------------------|
| Classifiers trained | K | 1 |
| Probabilities sum to 1? | No (each is independent) | Yes |
| Better when | Classes are clearly separated | Classes overlap; well-calibrated probs needed |

In [ ]:
# Q17 — Multiclass logistic regression
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_tr_ir, X_te_ir, y_tr_ir, y_te_ir = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)
sc_iris = StandardScaler()
X_tr_ir_s = sc_iris.fit_transform(X_tr_ir)
X_te_ir_s  = sc_iris.transform(X_te_ir)

for strategy in ['ovr', 'multinomial']:
    m = LogisticRegression(multi_class=strategy, max_iter=1000, random_state=42)
    m.fit(X_tr_ir_s, y_tr_ir)
    pred = m.predict(X_te_ir_s)
    prob = m.predict_proba(X_te_ir_s)
    acc  = accuracy_score(y_te_ir, pred)
    print(f'Strategy: {strategy:<14} | Accuracy: {acc:.4f}')
    print(f'  Sample probs (3 classes, sum to 1): {prob[0].round(4)} → sum={prob[0].sum():.4f}')
    print()

print('Both achieve high accuracy on Iris — it is a well-separated dataset.')
print('Softmax probabilities always sum to exactly 1.')

## Q18. What is the decision boundary in Logistic Regression?

**Answer:**

The **decision boundary** is the line (or hyperplane) that separates class 0 from class 1 predictions.

**How it's defined:**
- P(y=1) = 0.5 exactly when sigmoid input z = 0
- z = θ₀ + θ₁x₁ + θ₂x₂ = 0 → this is a linear equation → a straight line

**Key properties:**
- Logistic Regression creates a **linear** decision boundary
- Points above the line → class 1; below → class 0
- **Cannot** separate non-linearly separable data (e.g., XOR problem)
- Fix: add polynomial features manually, or use non-linear classifiers (SVM, trees)

**Can we change the threshold?**
Yes! Instead of 0.5, use:
- Lower threshold (e.g., 0.3) → more sensitive, higher recall (catches more positives)
- Higher threshold (e.g., 0.7) → more specific, higher precision (fewer false alarms)

In [ ]:
# Q18 — Visualize decision boundary + effect of changing threshold
from sklearn.datasets import make_classification

X_db, y_db = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                  n_informative=2, random_state=42, n_clusters_per_class=1)
log_db = LogisticRegression(max_iter=1000)
log_db.fit(X_db, y_db)

# Create mesh grid
x1_min, x1_max = X_db[:,0].min()-1, X_db[:,0].max()+1
x2_min, x2_max = X_db[:,1].min()-1, X_db[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                     np.linspace(x2_min, x2_max, 200))
Z = log_db.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Standard threshold 0.5
for ax, threshold, title in zip(axes, [0.5, 0.3],
                                  ['Default threshold=0.5', 'Lower threshold=0.3 (higher recall)']):
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu', levels=50)
    ax.contour(xx, yy, Z, levels=[threshold], colors='black', linewidths=2)
    scatter = ax.scatter(X_db[:,0], X_db[:,1], c=y_db, cmap='RdBu', edgecolor='k', alpha=0.7)
    pred = (log_db.predict_proba(X_db)[:,1] >= threshold).astype(int)
    from sklearn.metrics import recall_score, precision_score
    rec  = recall_score(y_db, pred)
    prec = precision_score(y_db, pred)
    ax.set_title(f'{title}\nPrecision={prec:.3f}, Recall={rec:.3f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Decision Boundary — Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 4 — Evaluation Metrics Interview Questions

## Q19. When would you choose RMSE over MAE? When would you choose MAE?

**Answer:**

| Metric | Use When |
|--------|----------|
| **RMSE** | Large errors are disproportionately costly (e.g., structural engineering, medical dosing) |
| **MAE** | All errors should be treated equally; data has outliers you don't want to over-weight |

**Key relationship:** RMSE ≥ MAE always. The bigger the gap, the more large errors are present.

**Practical tip:** Report both — if RMSE >> MAE, you have significant outlier errors to investigate.

**MAPE** is useful when you need a % error metric (e.g., "our forecast is off by 8% on average") — but avoid it when actuals can be 0.

In [ ]:
# Q19 — Show when RMSE >> MAE indicates outlier errors
y_true_met = np.array([100, 120, 110, 130, 115, 125, 105, 118])

scenarios = {
    'Small uniform errors': y_true_met + np.random.uniform(-5, 5, 8),
    'One large error'     : np.array([105, 125, 108, 135, 112, 122, 102, 400]),  # last one way off
    'Outlier in data'     : np.array([105, 125, 108, 135, 112, 122, 102, 120]),
}

print(f'{"Scenario":<25} {"MAE":>8} {"RMSE":>8} {"RMSE-MAE":>10} {"Interpretation"}')
print('-' * 80)
for name, y_pred_met in scenarios.items():
    mae_v  = mean_absolute_error(y_true_met, y_pred_met)
    rmse_v = np.sqrt(mean_squared_error(y_true_met, y_pred_met))
    interp = 'Large errors present!' if rmse_v - mae_v > 50 else 'Errors roughly uniform'
    print(f'{name:<25} {mae_v:>8.2f} {rmse_v:>8.2f} {rmse_v-mae_v:>10.2f}  {interp}')

## Q20. What is R²? What does R²=0 and R²<0 mean?

**Answer:**

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

| R² | Meaning |
|----|--------|
| 1.0 | Perfect predictions |
| 0.85 | Model explains 85% of variance — good |
| 0.0 | Model is no better than just predicting the mean |
| < 0 | Model is **worse** than predicting the mean — something is very wrong |

**Limitation of R²:** Adding any feature always increases R² (even irrelevant ones!) → use **Adjusted R²** when comparing models with different numbers of features:

$$\text{Adj } R^2 = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$

where n = samples, p = number of features. Adjusted R² penalizes for adding useless features.

In [ ]:
# Q20 — R² and Adjusted R²
from sklearn.datasets import make_regression

X_r2, y_r2 = make_regression(n_samples=100, n_features=5, noise=10, random_state=42)
X_r2_tr, X_r2_te, y_r2_tr, y_r2_te = train_test_split(X_r2, y_r2, test_size=0.2, random_state=42)

m_r2 = LinearRegression().fit(X_r2_tr, y_r2_tr)
pred_r2 = m_r2.predict(X_r2_te)

r2   = r2_score(y_r2_te, pred_r2)
n, p = len(y_r2_te), X_r2_te.shape[1]
adj_r2 = 1 - (1-r2) * (n-1) / (n-p-1)

print(f'n={n}, p={p} features')
print(f'R²          = {r2:.4f}')
print(f'Adjusted R² = {adj_r2:.4f}  (penalizes for {p} features)')

# Show R² decreases when adding random noise features
print('\nEffect of adding random (useless) features:')
print(f'{"Extra noise features":<25} {"R²":>8} {"Adj R²":>10}')
for extra in [0, 5, 20, 50]:
    noise = np.random.randn(X_r2_tr.shape[0], extra) if extra > 0 else np.empty((X_r2_tr.shape[0], 0))
    noise_te = np.random.randn(X_r2_te.shape[0], extra) if extra > 0 else np.empty((X_r2_te.shape[0], 0))
    X_aug_tr = np.hstack([X_r2_tr, noise]) if extra > 0 else X_r2_tr
    X_aug_te = np.hstack([X_r2_te, noise_te]) if extra > 0 else X_r2_te
    m_aug = LinearRegression().fit(X_aug_tr, y_r2_tr)
    pred_aug = m_aug.predict(X_aug_te)
    r2_aug  = r2_score(y_r2_te, pred_aug)
    p_aug   = 5 + extra
    adj_r2_aug = 1 - (1-r2_aug)*(n-1)/(n-p_aug-1)
    print(f'{extra:<25} {r2_aug:>8.4f} {adj_r2_aug:>10.4f}')

print('\nR² keeps going up with more features; Adjusted R² drops — it knows the extras are useless.')

## Q21. What is cross-validation? Why is it better than a single train/test split?

**Answer:**

**Cross-validation** splits the data into K folds, trains on K-1 folds, evaluates on the remaining fold, and repeats K times. Final score = average of K test scores.

**Why better than single split?**

| Problem with single split | CV solution |
|--------------------------|-------------|
| Test set might be "lucky" (easy samples) | Averages over K different test sets |
| High variance in evaluation | Much lower variance — more stable estimate |
| Wastes data (only 80% for training) | Every sample appears in the test set once |

**Common choices:**
- K=5: good default
- K=10: more reliable, more compute
- Leave-One-Out (LOO): K=n — most reliable, very slow on large datasets

In [ ]:
# Q21 — Cross-validation vs single split
housing = fetch_california_housing()
X_cv = housing.data
y_cv = housing.target
scaler_cv = StandardScaler()
X_cv_s = scaler_cv.fit_transform(X_cv)

# Single split — run 5 times with different random seeds
single_scores = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X_cv_s, y_cv, test_size=0.2, random_state=seed)
    m = Ridge(alpha=1.0).fit(X_tr, y_tr)
    single_scores.append(r2_score(y_te, m.predict(X_te)))

# 5-fold and 10-fold cross-validation
cv5_scores  = cross_val_score(Ridge(alpha=1.0), X_cv_s, y_cv, cv=5,  scoring='r2')
cv10_scores = cross_val_score(Ridge(alpha=1.0), X_cv_s, y_cv, cv=10, scoring='r2')

print('Single split (10 different random seeds):')
print(f'  Scores : {[round(s,4) for s in single_scores]}')
print(f'  Mean   : {np.mean(single_scores):.4f}')
print(f'  Std Dev: {np.std(single_scores):.4f}  ← high variance')

print('\n5-fold CV:')
print(f'  Scores : {cv5_scores.round(4)}')
print(f'  Mean   : {cv5_scores.mean():.4f} ± {cv5_scores.std():.4f}')

print('\n10-fold CV:')
print(f'  Scores : {cv10_scores.round(4)}')
print(f'  Mean   : {cv10_scores.mean():.4f} ± {cv10_scores.std():.4f}')

---
# Part 5 — Quick Cheat Sheet

## Regression Models Summary

| Model | Penalty | Zeros out features? | Best for |
|-------|---------|---------------------|----------|
| Linear Regression | None | No | Clean data, low dimensions |
| Ridge (L2) | λ Σwⱼ² | No | Multicollinearity; all features matter |
| Lasso (L1) | λ Σ\|wⱼ\| | **Yes** | Many irrelevant features; sparse model |
| ElasticNet | L1 + L2 | **Yes** | Correlated features + irrelevant ones |

---

## Metrics Quick Reference

### Regression Metrics
| Metric | Formula | Units | Outlier Sensitive? |
|--------|---------|-------|-------------------|
| MAE | mean(\|y-ŷ\|) | Same as y | No |
| MSE | mean((y-ŷ)²) | y² | Yes |
| RMSE | √MSE | Same as y | Yes |
| R² | 1 - SSres/SStot | None (0–1) | Moderate |
| MAPE | mean(\|y-ŷ\|/\|y\|)×100 | % | No |

### Classification Metrics
| Metric | Formula | Use when |
|--------|---------|----------|
| Accuracy | (TP+TN)/N | Balanced classes |
| Precision | TP/(TP+FP) | FP is costly (spam filter) |
| Recall | TP/(TP+FN) | FN is costly (disease detection) |
| F1 | 2PR/(P+R) | Imbalanced + both FP/FN matter |
| AUC-ROC | Area under ROC curve | Imbalanced; threshold-independent |

---

## Most Common Interview Questions — Quick Answers

| Question | One-line Answer |
|----------|----------------|
| Why sigmoid in logistic regression? | Maps any real number to (0,1) as a probability |
| Why log loss not MSE for classification? | MSE creates non-convex cost → gradient descent gets stuck |
| Ridge vs Lasso? | Ridge shrinks, Lasso zeros out — use Lasso for feature selection |
| What does AUC=0.5 mean? | Model is no better than random guessing |
| Can R² be negative? | Yes — when model is worse than predicting the mean |
| What does gradient descent do? | Iteratively updates parameters to minimize cost function |
| What is multicollinearity? | Features correlated → unstable coefficients → use Ridge or PCA |
| Precision vs Recall tradeoff? | Lowering threshold → higher recall, lower precision |
| Why scale features for regression? | Coefficients and gradient descent are sensitive to feature scale |